In [1]:
import json
import re
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

from pydantic import BaseModel

from typing import List

load_dotenv() 

True

In [2]:
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

str_path_pc = "C:\\Users\\lpasquarelli\\OneDrive - BUSINESS INTEGRATION PARTNERS SPA\\llmwiki"
alternative_mac_personal_path = "/Users/luco/llmwiki"

different_path: list[str] = [str_path_pc, alternative_mac_personal_path]

CHUNKS_DIR = Path(different_path[1]) / "chunks"
OBSIDIAN_DIR = Path(different_path[1]) / timestamp

OBSIDIAN_DIR.mkdir(parents=True,exist_ok=True)

In [3]:
MODEL_NAME = "openai:gpt-5.4-mini"
#OBSIDIAN_DIR = Path(r"/Users/luco/llmwiki/")
RAW_DIR = CHUNKS_DIR 
WIKI_DIR = OBSIDIAN_DIR / "AI Wiki"
PULSES_DIR = WIKI_DIR / "pulses"
PROJECTS_DIR = WIKI_DIR / "projects"
TECHNOLOGIES_DIR = WIKI_DIR / "technologies"
INDEX_FILE = WIKI_DIR / "index.md"

AGENT_WIKI = WIKI_DIR / "agent_wiki"
#prova

In [4]:
class TechnologyPulse(BaseModel):
    tech_id: str
    date: str
    new_skill: str

class ProjectPulse(BaseModel):
    project_id: str
    date: str
    current_state: str

class Pulse(BaseModel):
    pulse_id: str
    summary: str
    projects: list[ProjectPulse]
    technologies: list[TechnologyPulse]

class Pulses(BaseModel):
    pulses: list[Pulse]
    projects_list: List[str] = []
    technologies_list: List[str] = []  

In [5]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage


model = init_chat_model(
    MODEL_NAME
)

structured_llm = model.with_structured_output(Pulses)

SYSTEM_PROMPT = """You are a deterministic knowledge base extractor working on daily work reports.

Your goal is to extract structured nodes that will populate a personal professional 
knowledge base tracking project evolution, technologies used, and skills demonstrated.

## Output nodes

### PULSE
One pulse per day. Captures what problem was being solved, the approach taken, 
and key technical decisions. The pulse_id must be the date in YYYY-MM-DD format.

### PROJECTS
Each project has an id (the canonical project name) and a brief description of its 
state on that day. Projects are long-lived initiatives, not one-off tasks.
Avoid the use of slashes (/) in project names, since the names will be used to build pages in markdown later on.
Furthermore, try to keep projects toghether, meaning that projects are MACRO activities, and many times what seems different is in fact part of the same project 
and just a different step probably. So avoid splitting the activities but keep it under the same umbrella. Try to actual keep the number of projects to a minimum.

### TECHNOLOGIES
Each technology has an id (the canonical technology name) and a brief description 
of the specific skill learned or applied on that day.

## Technology classification rules — read carefully

A TECHNOLOGY must be a real, reusable software tool, library, framework, platform, 
or infrastructure service that exists independently of this project.

Valid technologies: pandas, scikit-learn, FastAPI, Databricks, VS Code, 
   Azure Functions, spaCy, Delta Lake, PySpark, pytest, Docker, Git

NOT technologies — do not add these:
- Project-specific artifacts: scripts, notebooks, modules, config files, 
  custom functions, field names, or internal identifiers (e.g. "LOOKUP_DY365", 
  "string_utils.py", "validation_engine")
- Concepts or processes: "normalization", "ground truth", "evaluation", 
  "data quality", "debugging"
- Generic tool categories without specificity: "notebook", "script", "pipeline", 
  "model" — always name the actual tool (e.g. "Databricks notebook", "scikit-learn 
  pipeline", "OpenAI GPT-4")

When in doubt, ask yourself: "Can I install or import this?" If no → it is not a technology.

## Vocabulary reuse rules

The knowledge base maintains canonical vocabularies:
- projects: {projects}
- technologies: {technologies}

Rules:
1. Always prefer an existing canonical name over creating a new one.
2. Only create a new entry if the concept is genuinely absent — not a variation, 
   alias, or abbreviation of something existing.
3. When you create a new entry, include it in projects_list / technologies_list 
   so the vocabulary stays up to date.
4. Use consistent casing: PascalCase for proper tool names (Databricks, VS Code), 
   lowercase for libraries (pandas, spacy, fastapi).

## Output language

Write all summaries, current_state, and new_skill fields in the same language 
as the input report. Do not switch languages mid-output.

## Field guidance

- pulse.summary: 2–4 sentences. Cover (1) the main problem tackled, (2) the 
  approach or solution attempted, (3) any open issues or decisions deferred. 
  Avoid vague filler like "I worked on X" — be specific about what changed.
- project.current_state: one sentence describing where the project stands on 
  this day, focusing on blockers, progress, or next steps.
- technology.new_skill: one sentence on what was learned or applied, specific 
  to that day's work.
"""

def build_prompt(report: str, projects: list[str] = [], technologies: list[str] = []) -> list:
    system = SYSTEM_PROMPT.format(
        projects=json.dumps(projects or []),
        technologies=json.dumps(technologies or [])
    )
    return [
        SystemMessage(content=system),
        HumanMessage(content=report),  # raw string, never parsed for {variables}
    ]

In [6]:
def list_raw_files() -> list[Path]:
    list_files = sorted(RAW_DIR.rglob("*.md"))
    if any(list_files):
        return list_files
    else:
        raise FileNotFoundError(f"No markdown files found in {RAW_DIR}")

def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8") if path.exists() else ""

In [7]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError

TIMEOUT_SECONDS = 120

CHUNKS_NUMBER= 1000

source_files = list_raw_files()
projects: List[str] = []
technologies: List[str] = []  

responses= []

#config = {"configurable": {"thread_id": "1234567890"}} 
for i, file in enumerate(source_files):
    content = read_text(file)
    prompt = build_prompt(content, projects, technologies)
    
    try:
        print(f"Processing: {file}")
        with ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(structured_llm.invoke, prompt)
            response = future.result(timeout=TIMEOUT_SECONDS)
        responses.append(response)
        projects.extend(response.projects_list)
        projects = list(set(projects))  # Remove duplicates
        technologies.extend(response.technologies_list)
        technologies = list(set(technologies))  # Remove duplicates
    
    except TimeoutError:
        print(f"Timeout processing file {file}")    
    except Exception as e:
        print(f"Error processing file {file}: {e}")
    if i == CHUNKS_NUMBER:    
        break
print(f"Processed {i+1} files")
    

Processing: /Users/luco/llmwiki/chunks/chunk_0.md
Processing: /Users/luco/llmwiki/chunks/chunk_1.md
Processing: /Users/luco/llmwiki/chunks/chunk_10.md
Processing: /Users/luco/llmwiki/chunks/chunk_2.md
Processing: /Users/luco/llmwiki/chunks/chunk_3.md
Processing: /Users/luco/llmwiki/chunks/chunk_4.md
Processing: /Users/luco/llmwiki/chunks/chunk_5.md
Processing: /Users/luco/llmwiki/chunks/chunk_6.md
Processing: /Users/luco/llmwiki/chunks/chunk_7.md
Processing: /Users/luco/llmwiki/chunks/chunk_8.md
Processing: /Users/luco/llmwiki/chunks/chunk_9.md
Processed 11 files


In [8]:
with open("response.json", "w") as f:
    json.dump(response.model_dump(), f, indent=2)

In [9]:
with open("responses.json","w") as f:
    json.dump([r.model_dump() for r in responses], f, indent=2)

In [10]:

# BUILDING MARKDOWN PAGES UTILITY FUNCTIONS
def render_pulse_md(pulse: Pulse):
    projects =  [p for p in pulse.projects]
    technologies = [t for t in pulse.technologies]

    render_technologies = "\n".join([f"- {t.tech_id}: {t.new_skill}" for t in technologies])
    render_projects = "\n".join([f"- {p.project_id}: {p.current_state}" for p in projects])
    return f"### {pulse.pulse_id} \n {pulse.summary} \n **Projects:** \n {render_projects} \n **Technologies:** \n {render_technologies}"


from collections import defaultdict

def sanitize_filename(name: str) -> str:
    return re.sub(r'[\\/:*?"<>|]', '_', name)

def init_pulses_page(page_path: Path) -> None:
    """Crea la pagina con il titolo se non esiste già."""
    if page_path.exists():
        return
    page_path.parent.mkdir(parents=True, exist_ok=True)
    title = page_path.stem  # es. "2026-01"
    page_path.write_text(f"# Pulses {title}\n\n", encoding="utf-8")

def append_pulse_to_page(page_path: Path, pulse: Pulse) -> None:
    """Appende il markdown di un pulse alla pagina."""
    with page_path.open("a", encoding="utf-8") as f:
        f.write(render_pulse_md(pulse) + "\n\n")

def write_pulses(pulses: list[Pulse], page_path: Path) -> None:
    # raggruppa per mese (YYYY-MM)
    by_month: dict[str, list[Pulse]] = defaultdict(list)
    for p in pulses:
        month_year = p.pulse_id[:7]   # "2026-01-26" -> "2026-01"
        by_month[month_year].append(p)

    for month_year, month_pulses in by_month.items():
        page = page_path / f"{month_year}.md"
        init_pulses_page(page)
        # ordina cronologicamente
        for pulse in sorted(month_pulses, key=lambda x: x.pulse_id):
            append_pulse_to_page(page, pulse)

# BUILD UTILITIES FOR PROJECTS AND TECHNOLOGIES PAGES 
def init_page(name:str, parent_path:Path):
    safe_name = sanitize_filename(name)
    page_path = parent_path / f"{safe_name}.md"
    print(page_path)
    if page_path.exists():
        print("already exists")
        return page_path
    page_path.parent.mkdir(parents=True, exist_ok=True)
    return page_path


def list_render_md(bullet_ids: List[str],page_path: Path):
    text = "\n - ".join(bullet_ids)
    with page_path.open("a", encoding="utf-8") as f:
        f.write("\n - ")
        f.write(text)

def nested_list_render_md(bullet_ids: List[List[str]],page_path: Path):
    lines_text = []
    for couple in bullet_ids:
        text = "on date: ".join(couple)
        lines_text.append(text)
    final_text = "\n - ".join(lines_text)
    with page_path.open("a", encoding="utf-8") as f:
        f.write("\n - ")
        f.write(final_text)    


def dict_render_md(pulses_dict: dict,page_path=TECHNOLOGIES_DIR):
    for key,values_list in pulses_dict.items():
        #initialize pages and append all the skills to each page
        key_path = init_page(name=key,parent_path=page_path)
        nested_list_render_md(values_list,key_path)

def write_util(pulses: List[Pulse], file_name: str = "technologies", type:str = "technologies", parent_path: Path = WIKI_DIR, tech_or_proj: Path = TECHNOLOGIES_DIR):
    bullet_set = set()
    pulse_dict = defaultdict(list)
    for p in pulses:
        if type == "technologies":
            for t in p.technologies:
                pulse_dict[t.tech_id].append([t.new_skill,t.date])
        elif type == "projects":
            for z in p.projects:
                pulse_dict[z.project_id].append([z.current_state,z.date])
        pulse_list = [key for key in pulse_dict.keys()]        
        bullet_set.update(pulse_list)
    bullet_list = list(bullet_set)    
    page_path = init_page(name=file_name,parent_path=parent_path)
    list_render_md(bullet_ids=bullet_list,page_path=page_path)
    dict_render_md(pulses_dict=pulse_dict,page_path=tech_or_proj)


In [11]:
# actually write the markdown

PULSES_DIR.mkdir(parents=True, exist_ok=True)
pulses = []
for response in responses:
    pulses.extend(response.pulses)
print(pulses)
write_pulses(pulses,page_path=PULSES_DIR)
write_util(pulses, file_name="technologies", type="technologies", parent_path=WIKI_DIR, tech_or_proj=TECHNOLOGIES_DIR)
write_util(pulses, file_name="projects", type="projects", parent_path=WIKI_DIR, tech_or_proj=PROJECTS_DIR)

[Pulse(pulse_id='2026-02-02', summary="Ho verificato che l'evaluation dei controlli non è automatica ma manuale, quindi il focus è stato sul consolidamento delle logiche di validazione per usi commerciali. Ho corretto il controllo `coerenza_residenza_diversa_fornitura` riscrivendone la logica e ho evidenziato le aree ancora aperte su Dichiarazioni, Usi promiscui e Firma, soprattutto perché alcuni campi non risultano estratti dal flusso OCR/LLM. Rimane da chiarire come trattare i controlli facoltativi e come confrontare correttamente la firma digitale rispetto alla ground truth.", projects=[ProjectPulse(project_id='Estensione validazione usi commerciali', date='2026-02-02', current_state='La validazione dei moduli commerciali è quasi pronta ma restano da integrare Dichiarazioni, Usi promiscui e Firma, con diversi dubbi su campi facoltativi e campi non estratti.')], technologies=[TechnologyPulse(tech_id='Databricks', date='2026-02-02', new_skill="Ho chiarito che l'evaluation dei controll

## Second section

- 1 Build a learning map for every technology and a timeline for every project

In [12]:
from pydantic import Field

In [13]:
# 1. Define the schema
class MarkdownOutputTech(BaseModel):
    """The formatted markdown content."""
    tech_id: str = Field(description="Name of the technology")
    content: str = Field(description="The full, cleaned markdown document content.")

In [14]:
TECH_PROMPT = """You are a technical knowledge curator.

You receive a raw technology page from a personal knowledge base. 
The page contains brief, context-specific bullet points extracted from daily work reports, with also the date.

Your job is to produce a structured, enriched page for that technology that:
1. Synthesizes the raw observations into a coherent overview, if something is not very clear because context is lacking, omit it
2. Identifies the specific patterns and concepts the person is actually applying
3. Complements the bullet points with the knowledge you have about the specific technology and skills at hand
4. Order the output points in such a way that it becomes clear the evolution of the knowledge for that given technology following the timeline evolution that the dates help to build

## Output structure

### overview
2 sentences. Describe what the technology is, why it matters, and how this 
specific person is using it based on the raw bullets. Be concrete — avoid 
generic definitions that ignore the usage context, but also you have to be clear, especially if the usage is not understandable without the context.


### key_concepts
For each pattern identified, extract the underlying technical concept 
For each concept provide:
- concept: the canonical name of the concept
- description: 2-3 sentences explaining what it is and why it matters

### learning path
Identify and narrate the learning journey that the person went through, identifying each milestone and narrating it in a structural way.
Keep the dates, but with the following formatting: "on date: YYYY-MM-DD" ans keep it sorted by date.


## Rules
- Write in the same language as the input page.
- Never invent usage that is not supported by the raw bullets.
- key_concepts must be real, importable/callable constructs — not vague ideas.
- next_steps must feel personally relevant, not generic "learn pandas" advice.
- Be specific and technical. This is for a professional knowledge base, not a blog post.
"""

def build_tech_prompt(tech_page_content: str) -> list:
    return [
        SystemMessage(content=TECH_PROMPT),
        HumanMessage(content=tech_page_content),
    ]

In [15]:
model2 = init_chat_model(
    "openai:gpt-5.4-mini"
)

structured_llm2 = model2.with_structured_output(MarkdownOutputTech)


In [16]:
list_tech = TECHNOLOGIES_DIR.rglob("*.md")
CONSTANT= 3
tech_pages = []
for i, tech in enumerate(list_tech):
    # if i == CONSTANT:
    file = read_text(tech)
    tech_prompt = build_tech_prompt(file)
    tech_page = structured_llm2.invoke(tech_prompt)
    tech_pages.append(tech_page)
   

In [17]:
with open("tech_pages_llm.json", "w") as f:
    json.dump([tech_page.model_dump() for tech_page in tech_pages], f, indent=2)

In [ ]:
AGENT_WIKI.mkdir(parents=True, exist_ok=True)

AGENT_TECHNOLOGIES = AGENT_WIKI / "TECHNOLOGIES"
AGENT_TECHNOLOGIES.mkdir(parents=True, exist_ok=True)

In [ ]:
for tech_page in tech_pages:
    new_path = init_page(name=tech_page.tech_id,parent_path=AGENT_TECHNOLOGIES)
    with new_path.open("a", encoding="utf-8") as f:
        f.write(tech_page.content)


/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/Databricks.md
/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/OCR e lettura strutturata dei documenti.md
/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/Git e flusso di lavoro su repository Python_notebook.md
/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/VS Code + Databricks.md
/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/FastAPI.md


## Third section

- 1 Build a learning map for every project with a timeline


In [20]:
class MarkdownOutputProject(BaseModel):
    """The formatted markdown content for a project."""
    project_id: str = Field(description="Name of the project")
    content: str = Field(description="The full, cleaned markdown document content.")


In [21]:
PROJECT_PROMPT = """You are a project knowledge curator.

You receive a raw project page from a personal knowledge base.
The page contains brief, context-specific bullet points extracted from daily work reports, with also the date.

Your job is to produce a structured, enriched page for that project that:
1. Synthesizes the raw observations into a coherent overview of the project's evolution
2. Identifies the key phases, milestones, and decisions that shaped the project
3. Highlights blockers encountered and how they were resolved (or left open)
4. Orders the output points to clearly show the project's timeline

## Output structure

### overview
3 sentences. Describe what the project is about and what its goal is, based on the raw bullets.
Be concrete — avoid generic descriptions that ignore the actual work being done.

### key_phases
For each identifiable phase or milestone in the project, provide:
- phase: a short label for the phase
- description: 2 sentences explaining what was tackled, what was decided, and what the outcome was

### timeline
Narrate the evolution of the project as a structured timeline.
Keep the dates, formatted as "on date: YYYY-MM-DD", sorted chronologically.
For each date entry, describe the state of the project and what progressed or changed.

## Rules
- Write in the same language as the input page.
- Never invent progress or decisions not supported by the raw bullets.
- Be specific and technical. This is for a professional knowledge base, not a blog post.
- If context is missing or unclear, omit rather than speculate.
"""

def build_project_prompt(project_page_content: str) -> list:
    return [
        SystemMessage(content=PROJECT_PROMPT),
        HumanMessage(content=project_page_content),
    ]


In [22]:
model3 = init_chat_model("openai:gpt-5.4-mini")

structured_llm3 = model3.with_structured_output(MarkdownOutputProject)


In [23]:
list_proj = PROJECTS_DIR.rglob("*.md")
project_pages = []
for proj in list_proj:
    file = read_text(proj)
    proj_prompt = build_project_prompt(file)
    proj_page = structured_llm3.invoke(proj_prompt)
    project_pages.append(proj_page)


In [24]:
with open("project_pages_llm.json", "w") as f:
    json.dump([proj_page.model_dump() for proj_page in project_pages], f, indent=2)


In [ ]:
AGENT_PROJECTS = AGENT_WIKI / "projects"
AGENT_PROJECTS.mkdir(parents=True, exist_ok=True)

for proj_page in project_pages:
    new_path = init_page(name=proj_page.project_id, parent_path=AGENT_PROJECTS)
    with new_path.open("a", encoding="utf-8") as f:
        f.write(proj_page.content)


/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/projects/Validazione moduli commerciali e usi agevolati.md
/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/projects/Project.md
/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/projects/Progetto validazione ed estrazione tabellare IVA.md
/Users/luco/llmwiki/20260627_120124/AI Wiki/agent_wiki/projects/OCR Batch _ processamento documenti su GCP e Power BI.md
